In [ ]:
import pandas as pd
import numpy as np

WEEK 7: OUTLIER DETECTION AND DATA QUALITY

## Objective
Identify extreme values in property sales data using the Interquartile Range (IQR) method.

The analysis focuses on:
- ClosePrice
- LivingArea
- DaysOnMarket

Outliers will be flagged rather than permanently deleted. A separate filtered dataset will be created for reliable market analysis.

In [ ]:
# Load Week 6 dataset
df = pd.read_csv(
    "../sold_residential_with_school_districts.csv",
    low_memory=False
)

print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
# Check target columns
target_columns = ["ClosePrice", "LivingArea", "DaysOnMarket"]

df[target_columns].info()

# Count missing values
df[target_columns].isna().sum()

In [ ]:
# Calculate IQR thresholds for each variable

iqr_results = {}

for col in target_columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    iqr_results[col] = {
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Lower Bound": lower,
        "Upper Bound": upper
    }

# Display results
iqr_table = pd.DataFrame(iqr_results).T
iqr_table.round(2)

In [ ]:
# Create IQR outlier flags

for col in target_columns:
    lower = iqr_results[col]["Lower Bound"]
    upper = iqr_results[col]["Upper Bound"]

    df[f"{col}_outlier_flag"] = (
        (df[col] < lower) | (df[col] > upper)
    ).astype(int)

# Flag invalid nonpositive values
for col in target_columns:
    df[f"{col}_invalid_flag"] = (
        df[col].notna() & (df[col] <= 0)
    ).astype(int)

# Count flagged records
flag_summary = pd.DataFrame({
    "IQR Outliers": [
        df[f"{col}_outlier_flag"].sum()
        for col in target_columns
    ],
    "Invalid Values": [
        df[f"{col}_invalid_flag"].sum()
        for col in target_columns
    ]
}, index=target_columns)

flag_summary

In [ ]:
# Flag records with at least one IQR outlier
outlier_flag_columns = [
    f"{col}_outlier_flag" for col in target_columns
]

df["any_outlier_flag"] = (
    df[outlier_flag_columns].any(axis=1).astype(int)
)

# Flag records with at least one invalid value
invalid_flag_columns = [
    f"{col}_invalid_flag" for col in target_columns
]

df["any_invalid_flag"] = (
    df[invalid_flag_columns].any(axis=1).astype(int)
)

# Summary
print("Records with at least one IQR outlier:",
      f"{df['any_outlier_flag'].sum():,}")

print("Records with at least one invalid value:",
      f"{df['any_invalid_flag'].sum():,}")